<a href="https://www.kaggle.com/code/pavankumar960/diabetes-analysis-prediction-and-model-deployment?scriptVersionId=222765702" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# install missing libraries, make sure Internet is ON
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.2/62.2 MB 24.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.9/321.9 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 81.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.1 MB/s eta 0:00:00
  Attempting uninstall: markupsafe
    Found existing installation: MarkupSafe 3.0.2
    Uninstalling MarkupSafe-3.0.2:
      Successfully uninstalled MarkupSafe-3.0.2


## Importing Libraries

In [2]:
#Basic Libs
import pandas as pd
import numpy as np

#for Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

#for Model Training
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

#for Model Deployment
import pickle
import gradio as gr

import warnings
warnings.filterwarnings('ignore')


## Loading Dataset

In [3]:
#file path
file = '/kaggle/input/diabetes-prediction-in-america-dataset/diabetes_dataset.csv'

df = pd.read_csv(file)

print('Sucessfully loaded')

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/diabetes-prediction-in-america-dataset/diabetes_dataset.csv'

## EDA

In [ ]:
#Dataset Info
print('\nDataset Info:')
display(df.info())

#Sample Dataset
print('\nSample Dataset:')
display(df.head())

#Basic Statistics
print('\nBasic Statistics:')
display(df.describe())


In [ ]:
#checking for Nulls
print('\n Nulls in DataSet:')
display(df.isnull().sum())

#checking for Duplicates
print('\n Duplicates in DataSet:')
display(df.duplicated().sum())

### Analysis Notes
 No Null Values
 No Duplicates
 Target Column - Diabeties_Diagnosis

## Data Visualization

### Univariate Analysis

In [ ]:
#Univariate Analysis
print("Unique value counts per column:")
unique_counts = df.nunique()
print(unique_counts)

# For each column
for col in df.columns:
    print(f"\nColumn: {col}")
    num_unique = df[col].nunique()
    print(f"Number of unique values: {num_unique}")
    # If the unique values are few, display them:
    if num_unique <= 10:
        print("Unique values:")
        print(df[col].unique())
    else:
        print("Unique values list is too long to display.")

#Pie Chart Function
def plot_pie(data, column, title=None):
    counts = data.value_counts()
    plt.figure(figsize=(6,6))
    plt.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=140)
    plt.title(title if title else f'Distribution of {column}')
    plt.axis('equal')
    plt.show()

#Pie Charts for columns with few unique values
few_unique_cols = ['Gender', 'Ethnicity', 'Smoking_Status', 'Family_History_Diabetes', 
                   'Heart_Disease_History', 'Physical_Activity_Level', 'Stress_Level', 
                   'Medication_Use', 'Diabetes_Diagnosis', 'Alcohol_Consumption_Per_Week']

for col in few_unique_cols:
    if col in df.columns:
        plot_pie(df[col], col)

#For columns with many unique values, we bin them into groups before plotting

# 1. Age: Categorize as 'Young Adult', 'Adult', 'Senior'
# (Here we assume: <35 = Young Adult, 35-60 = Adult, >60 = Senior.)
df['Age_Group'] = pd.cut(df['Age'], bins=[0, 35, 60, df['Age'].max()], 
                           labels=['Young Adult', 'Adult', 'Senior'], right=False)
plot_pie(df['Age_Group'], 'Age_Group', title='Age Group Distribution')

# 2. BMI: Categorize as 'Low', 'Medium', 'High'
#(Using standard BMI ranges: <18.5 = Low, 18.5-25 = Medium, >=25 = High)
df['BMI_Group'] = pd.cut(df['BMI'], bins=[df['BMI'].min()-1, 18.5, 25, df['BMI'].max()+1], 
                           labels=['Low', 'Medium', 'High'])
plot_pie(df['BMI_Group'], 'BMI_Group', title='BMI Group Distribution')

# 3. Blood Pressure: Use equal-width binning into three groups: Low, Medium, High
df['BP_Group'] = pd.cut(df['Blood_Pressure'], bins=3, labels=['Low', 'Medium', 'High'])
plot_pie(df['BP_Group'], 'BP_Group', title='Blood Pressure Group Distribution')

# 4. Glucose_Level: For demonstration, use quantile-based binning into 3 groups.
df['Glucose_Group'] = pd.qcut(df['Glucose_Level'], q=3, labels=['Low', 'Medium', 'High'])
plot_pie(df['Glucose_Group'], 'Glucose_Group', title='Glucose Level Group Distribution')

# 5. HbA1c: Use quantile-based binning into 3 groups.
df['HbA1c_Group'] = pd.qcut(df['HbA1c'], q=3, labels=['Low', 'Medium', 'High'])
plot_pie(df['HbA1c_Group'], 'HbA1c_Group', title='HbA1c Group Distribution')

# 6. Insulin_Resistance: Use quantile-based binning into 3 groups.
df['IR_Group'] = pd.qcut(df['Insulin_Resistance'], q=3, labels=['Low', 'Medium', 'High'])
plot_pie(df['IR_Group'], 'IR_Group', title='Insulin Resistance Group Distribution')

# 7. Sleep_Hours_Per_Night: Categorize as 'Short', 'Normal', 'Long'
#(For example, <6 hours = Short, 6-8 = Normal, >8 = Long)
df['Sleep_Group'] = pd.cut(df['Sleep_Hours_Per_Night'], bins=[df['Sleep_Hours_Per_Night'].min()-1, 6, 8, df['Sleep_Hours_Per_Night'].max()+1],
                           labels=['Short', 'Normal', 'Long'])
plot_pie(df['Sleep_Group'], 'Sleep_Group', title='Sleep Hours Group Distribution')

# 8. Income: Use quantile-based binning into 3 groups.
df['Income_Group'] = pd.qcut(df['Income'], q=3, labels=['Low', 'Medium', 'High'])
plot_pie(df['Income_Group'], 'Income_Group', title='Income Group Distribution')


### Bivariate Analysis

In [ ]:
# loading target 'Diabetes_Diagnosis'
target = 'Diabetes_Diagnosis'

# Bivariate Analysis for Numerical Variables
# Dropping Numerical Columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.drop(target)

# Boxplots
for col in numeric_cols:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=target, y=col, data=df)
    plt.title(f'{col} vs {target} (Boxplot)')
    plt.xlabel('Diabetes Diagnosis')
    plt.ylabel(col)
    plt.show()

# Violin plots
for col in numeric_cols:
    plt.figure(figsize=(8, 4))
    sns.violinplot(x=target, y=col, data=df)
    plt.title(f'{col} vs {target} (Violinplot)')
    plt.xlabel('Diabetes Diagnosis')
    plt.ylabel(col)
    plt.show()

# Bivariate Analysis for Categorical Variables
# Categorical column
categorical_cols = df.select_dtypes(include=['object']).columns

# CountPlot
for col in categorical_cols:
    plt.figure(figsize=(8, 4))
    sns.countplot(x=col, hue=target, data=df)
    plt.title(f'{col} vs {target} (Countplot)')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.legend(title=target)
    plt.show()

In [ ]:
# HeatMap
plt.figure(figsize=(10, 8))
corr = df.select_dtypes(include=['int64', 'float64']).corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:

# 2. Create Age bins to group the continuous Age into ranges
age_bins = [20, 30, 40, 50, 60, 70, 80]  # adjust these edges based on your data
age_labels = ['20-30', '30-40', '40-50', '50-60', '60-70', '70-80']
df['Age_bin'] = pd.cut(df['Age'], bins=age_bins, labels=age_labels, right=False)

# 3. Group by Age_bin, Gender, and Diabetes_Diagnosis.
# For demonstration, we will compute the mean for two numeric features: 'BMI' and 'Glucose_Level'.
# (Replace 'Glucose_Level' with the appropriate column name if different in your dataset.)
grouped = df.groupby(['Age_bin', 'Gender', 'Diabetes_Diagnosis'])[['BMI', 'Glucose_Level']].mean().reset_index()

# 4. Create a scatter plot using seaborn's FacetGrid to facet by Age_bin (rows) and Gender (columns).
g = sns.FacetGrid(grouped, row='Age_bin', col='Gender', hue='Diabetes_Diagnosis', 
                  margin_titles=True, height=3, aspect=1)
g.map(plt.scatter, 'BMI', 'Glucose_Level', s=80)

# 5. Annotate each point with the actual mean values.
def annotate(data, **kwargs):
    ax = plt.gca()
    for _, row in data.iterrows():
        # Format the label to display BMI and Glucose_Level with one decimal place.
        label = f"BMI: {row['BMI']:.1f}\nGluc: {row['Glucose_Level']:.1f}"
        ax.text(row['BMI'], row['Glucose_Level'], label, fontsize=8, ha='right', va='center')

g.map_dataframe(annotate)

# 6. Customize the plot: add a legend, axis labels, and a title.
g.add_legend(title="Diabetes Diagnosis\n(0 = No, 1 = Yes)")
g.set_axis_labels("Mean BMI", "Mean Glucose Level")
plt.subplots_adjust(top=0.92)
g.fig.suptitle("Scatter Plot of Mean BMI vs. Mean Glucose Level\nGrouped by Age Bin, Gender, and Diagnosis")

plt.show()


## Model Training

In [ ]:
# Dropping target from features
X = df.drop('Diabetes_Diagnosis', axis=1)
y = df['Diabetes_Diagnosis']

# Seperating numerical and categorical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

# Define preprocessing pipelines for numerical and categorical features
num_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine transformers using ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

# Build the model pipeline (preprocessing + classifier)
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(solver='liblinear'))  # You can change to another classifier
])

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Train the model
model_pipeline.fit(X_train, y_train)

# Evaluate the model on the test set
y_pred = model_pipeline.predict(X_test)
y_proba = model_pipeline.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("ROC AUC Score:", roc_auc_score(y_test, y_proba))


In [ ]:
# Get feature names after encoding
feature_names = num_cols + list(model_pipeline.named_steps['preprocessor'].named_transformers_['cat']
                                .named_steps['onehot'].get_feature_names_out(cat_cols))

# Extract model coefficients
coefs = model_pipeline.named_steps['classifier'].coef_[0]

# Create a DataFrame with feature names and their coefficients
feature_importance = pd.DataFrame({'Feature': feature_names, 'Weight': coefs})
feature_importance = feature_importance.sort_values(by='Weight', ascending=False)

# Display top 10 most influential features
print("Top 10 Features Influencing Diabetes Prediction:")
print(feature_importance.head(5))

In [ ]:
# Sort feature importance for better visualization
feature_importance = feature_importance.sort_values(by='Weight', ascending=True)

# Plot bar chart
plt.figure(figsize=(12, 6))
plt.barh(feature_importance['Feature'], feature_importance['Weight'], color='royalblue')

# Labels and title
plt.xlabel('Weight (Coefficient)', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Feature Importance (Logistic Regression)', fontsize=14)
plt.grid(axis='x', linestyle='--', alpha=0.6)

# Show plot
plt.show()


In [ ]:
# Save the pipeline to a file
with open('diabetes_model.pkl', 'wb') as f:
    pickle.dump(model_pipeline, f)


In [ ]:
# Load the saved model pipeline
with open('diabetes_model.pkl', 'rb') as f:
    model = pickle.load(f)

# Function for diabetes prediction
def predict_diabetes(
    Age=50,
    Gender="Male",
    Ethnicity="Caucasian",
    Income=50000,
    BMI=27.5,
    Blood_Pressure=120,
    Cholesterol=190,
    Exercise_Hours_Per_Week=3,
    Alcohol_Consumption_Per_Week=2,
    Smoking_Status="Never",
    Family_History_Diabetes=0,
    Glucose_Level=100,
    HbA1c=5.5,
    Insulin_Resistance=1,
    Heart_Disease_History=0,
    Physical_Activity_Level="Moderate",
    Fast_Food_Intake_Per_Week=2,
    Processed_Food_Intake_Per_Week=3,
    Daily_Caloric_Intake=2000,
    Sleep_Hours_Per_Night=7,
    Stress_Level="Moderate",
    Medication_Use=0
):
    # Create a dataframe from the input values
    input_data = pd.DataFrame({
        "Age": [Age],
        "Gender": [Gender],
        "Ethnicity": [Ethnicity],
        "Income": [Income],
        "BMI": [BMI],
        "Blood_Pressure": [Blood_Pressure],
        "Cholesterol": [Cholesterol],
        "Exercise_Hours_Per_Week": [Exercise_Hours_Per_Week],
        "Alcohol_Consumption_Per_Week": [Alcohol_Consumption_Per_Week],
        "Smoking_Status": [Smoking_Status],
        "Family_History_Diabetes": [Family_History_Diabetes],
        "Glucose_Level": [Glucose_Level],
        "HbA1c": [HbA1c],
        "Insulin_Resistance": [Insulin_Resistance],
        "Heart_Disease_History": [Heart_Disease_History],
        "Physical_Activity_Level": [Physical_Activity_Level],
        "Fast_Food_Intake_Per_Week": [Fast_Food_Intake_Per_Week],
        "Processed_Food_Intake_Per_Week": [Processed_Food_Intake_Per_Week],
        "Daily_Caloric_Intake": [Daily_Caloric_Intake],
        "Sleep_Hours_Per_Night": [Sleep_Hours_Per_Night],
        "Stress_Level": [Stress_Level],
        "Medication_Use": [Medication_Use]
    })
    
    # Predict the probability of diabetes
    probability = model.predict_proba(input_data)[:, 1][0]
    return probability

# Gradio interface
interface = gr.Interface(
    fn=predict_diabetes,
    inputs=[
        gr.Number(label="Age", value=50),
        gr.Dropdown(label="Gender", choices=["Male", "Female", "Other"], value="Male"),
        gr.Textbox(label="Ethnicity", placeholder="e.g., Hispanic, Caucasian, etc.", value="Caucasian"),
        gr.Number(label="Income", value=50000),
        gr.Number(label="BMI", value=27.5),
        gr.Number(label="Blood Pressure", value=120),
        gr.Number(label="Cholesterol", value=190),
        gr.Number(label="Exercise Hours Per Week", value=3),
        gr.Number(label="Alcohol Consumption Per Week", value=2),
        gr.Dropdown(label="Smoking Status", choices=["Never", "Former", "Current"], value="Never"),
        gr.Number(label="Family History of Diabetes (0 or 1)", value=0),
        gr.Number(label="Glucose Level", value=100),
        gr.Number(label="HbA1c", value=5.5),
        gr.Number(label="Insulin Resistance", value=1),
        gr.Number(label="Heart Disease History (0 or 1)", value=0),
        gr.Dropdown(label="Physical Activity Level", choices=["Low", "Moderate", "High"], value="Moderate"),
        gr.Number(label="Fast Food Intake Per Week", value=2),
        gr.Number(label="Processed Food Intake Per Week", value=3),
        gr.Number(label="Daily Caloric Intake", value=2000),
        gr.Number(label="Sleep Hours Per Night", value=7),
        gr.Dropdown(label="Stress Level", choices=["Low", "Moderate", "High"], value="Moderate"),
        gr.Number(label="Medication Use (0 or 1)", value=0)
    ],
    outputs=gr.Number(label="Diabetes Probability"),
    title="Diabetes Prediction",
    description="Enter patient details to predict the probability of having diabetes. Default values are set to approximate averages."
)

# Launch the Gradio app
interface.launch()
